# PRH audit: re-measuring the Aristotelian setup (Paper A, Step 1)

**What this does, end to end:**
1. Extracts features for the exact Huh et al. / Aristotelian model set on WIT-1024: 12 LLMs (BLOOMZ, OpenLLaMA, LLaMA 7B–65B) and 17 ViTs (ImageNet-21k, MAE, DINOv2, CLIP, CLIP-ft). Same pooling as their code: mean over tokens for LLMs, CLS per block for ViTs.
2. For every LLM × vision pair and every layer pair, under 3 preprocessings (`huh` = their exact clip + L2-normalize, `noclip`, `std` = per-dimension standardization), it computes:
   raw linear CKA (max over layer pairs), **Aristotelian calibrated CKA** (permutation null on the max, 200 permutations, α = 0.05), unbiased CKA (max, and **held-out layer selection**), mutual k-NN (raw and calibrated), plus effective rank at the chosen layers and the E1 predicted shrinkage.
3. Outlier diagnostics for every model and layer.
4. Trend analysis against LLM capability (bits per byte), figures, and a `summary.json`.

**How to run on an H100 (High-RAM runtime recommended):**
1. Leave `SMOKE_TEST = True` and choose *Runtime → Run all*. This takes about 5–10 minutes with 3 small LLMs and 2 ViTs, and checks that everything works.
2. Set `SMOKE_TEST = False` and *Run all* again for the full run (roughly 2–4 hours, mostly downloading ~300 GB of LLM weights; LLaMA-65B runs in 4-bit as in Huh et al.'s `--qlora`).
3. Upload the final `prh_audit_results.zip` (a few MB) back to Claude.

Everything is saved to Google Drive as it goes. If the runtime disconnects, just *Run all* again: finished models and pairs are skipped.

In [ ]:
# ============================== CONFIG ==============================
import os
SMOKE_TEST = True            # True: quick check with 3 small LLMs + 2 ViTs. False: full Aristotelian model set.
INCLUDE_LLAMA_30B = True     # bf16, ~65 GB of weights; fits an 80 GB H100
INCLUDE_LLAMA_65B = True     # 4-bit NF4 (Huh et al.'s --qlora); needs ~135 GB free disk to download
N_PERMS = 200                # permutations for the calibration null (Aristotelian paper: 200)
ALPHA = 0.05
TOPK = 10                    # mutual k-NN neighbours (Huh et al.: 10)
VARIANTS = ("huh", "noclip", "std")
SAVE_TO_DRIVE = True         # keep features & results on Drive so a disconnect loses nothing
SEED = 0

LLMS_FULL = ["bigscience/bloomz-560m", "bigscience/bloomz-1b1", "bigscience/bloomz-1b7", "bigscience/bloomz-3b",
             "bigscience/bloomz-7b1", "openlm-research/open_llama_3b", "openlm-research/open_llama_7b",
             "openlm-research/open_llama_13b", "huggyllama/llama-7b", "huggyllama/llama-13b",
             "huggyllama/llama-30b", "huggyllama/llama-65b"]
VISION_FULL = ["vit_tiny_patch16_224.augreg_in21k", "vit_small_patch16_224.augreg_in21k",
               "vit_base_patch16_224.augreg_in21k", "vit_large_patch16_224.augreg_in21k",
               "vit_base_patch16_224.mae", "vit_large_patch16_224.mae", "vit_huge_patch14_224.mae",
               "vit_small_patch14_dinov2.lvd142m", "vit_base_patch14_dinov2.lvd142m",
               "vit_large_patch14_dinov2.lvd142m", "vit_giant_patch14_dinov2.lvd142m",
               "vit_base_patch16_clip_224.laion2b", "vit_large_patch14_clip_224.laion2b",
               "vit_huge_patch14_clip_224.laion2b", "vit_base_patch16_clip_224.laion2b_ft_in12k",
               "vit_large_patch14_clip_224.laion2b_ft_in12k", "vit_huge_patch14_clip_224.laion2b_ft_in12k"]
FOUR_BIT = {"huggyllama/llama-65b"}
APPROX_GB = {"bigscience/bloomz-7b1": 16, "openlm-research/open_llama_13b": 27, "huggyllama/llama-13b": 27,
             "huggyllama/llama-30b": 66, "huggyllama/llama-65b": 132}

if SMOKE_TEST:
    LLMS = ["bigscience/bloomz-560m", "bigscience/bloomz-1b1", "bigscience/bloomz-1b7"]
    VISION = ["vit_tiny_patch16_224.augreg_in21k", "vit_small_patch14_dinov2.lvd142m"]
    N_PERMS = 50
else:
    LLMS = [m for m in LLMS_FULL if (INCLUDE_LLAMA_30B or m != "huggyllama/llama-30b")
            and (INCLUDE_LLAMA_65B or m != "huggyllama/llama-65b")]
    VISION = VISION_FULL

if os.environ.get("PRH_DRYRUN") == "1":          # local test only; never set on Colab
    from dryrun_support import DRY_LLMS, DRY_VISION
    LLMS, VISION, N_PERMS = DRY_LLMS, DRY_VISION, 20

def llm_batch_size(name):
    gb = APPROX_GB.get(name, 8)
    return 32 if gb <= 8 else 16 if gb <= 30 else 4

In [ ]:
# ============================== SETUP ==============================
import os, sys, subprocess, json, time, glob, shutil
DRYRUN = os.environ.get("PRH_DRYRUN") == "1"     # local test mode (tiny random models); ignore on Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm", "bitsandbytes", "accelerate",
                    "datasets", "sentencepiece", "protobuf"], check=False)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # optional: add HF_TOKEN in Colab secrets
    except Exception:
        pass

ROOT = os.environ.get("PRH_ROOT") or ("/content/drive/MyDrive/prh_audit" if (IN_COLAB and SAVE_TO_DRIVE) else "/content/prh_audit")
RUN = os.path.join(ROOT, "smoke" if SMOKE_TEST else "full")
FEAT_DIR, RES_DIR, MAT_DIR, FIG_DIR = (os.path.join(RUN, d) for d in ("features", "results", "matrices", "figs"))
for d in (FEAT_DIR, RES_DIR, MAT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

import torch, numpy as np, pandas as pd
torch.backends.cuda.matmul.allow_tf32 = False      # exact fp32 for the metrics
torch.backends.cudnn.allow_tf32 = False
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
print("run dir:", RUN)
print("free disk (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
LOG = os.path.join(RUN, "log.txt")
def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
    print(s); open(LOG, "a").write(s + "\n")

In [ ]:
# ============================== ENGINE ==============================
# ===================== Alignment engine (all layer pairs at once, GPU) =====================
# Every metric is computed for ALL (layer_x, layer_y) pairs with one matmul on flattened
# n x n matrices. Permutation nulls permute one side's matrices with the SAME permutation
# for every layer, so the max-over-layer-pairs aggregate is calibrated as in the Aristotelian paper.
import math, numpy as np, torch

VARIANTS = ("huh", "noclip", "std")
# huh    : clamp |x| at the per-sample 95th percentile (averaged), then L2-normalise each sample
#          (exactly Huh et al.'s prepare_features + compute_score(normalize=True))
# noclip : L2-normalise only
# std    : per-layer, per-dimension z-score over samples, then L2-normalise

def huh_clip(feats, q=0.95):
    """Huh et al. remove_outliers(exact=False): per-sample q-quantile of |x| over all layers & dims
    (linear interpolation, as torch.quantile), averaged over samples, then clamp. Uses kthvalue so it
    also works on tensors larger than torch.quantile's size limit."""
    a = feats.abs().flatten(start_dim=1); m = a.shape[1]
    pos = q * (m - 1); lo = int(math.floor(pos)); hi = int(math.ceil(pos)); w = pos - lo
    qs = []
    for i in range(a.shape[0]):
        v_lo = a[i].kthvalue(lo + 1).values
        v_hi = a[i].kthvalue(hi + 1).values if hi != lo else v_lo
        qs.append(v_lo + (v_hi - v_lo) * w)
    t = torch.stack(qs).mean()
    return feats.clamp(-t, t)

def preprocess(feats, variant):
    """feats: N x L x D float32 tensor -> list of L tensors (N x D), ready for metrics."""
    if variant == "huh":
        f = huh_clip(feats)
    elif variant == "noclip":
        f = feats
    elif variant == "std":
        f = (feats - feats.mean(0, keepdim=True)) / (feats.std(0, keepdim=True) + 1e-6)
    else:
        raise ValueError(variant)
    return [torch.nn.functional.normalize(f[:, l, :], dim=-1) for l in range(f.shape[1])]

def _center(K):
    m = K.mean(0)
    return K - m[None, :] - m[:, None] + m.mean()

class SideStats:
    """Per-model, per-variant matrices needed for every metric, flattened over layers."""
    def __init__(self, layers, k=10, half_idx=None):
        n = layers[0].shape[0]; self.n = n; self.L = len(layers); self.k = k
        Kc, Kt, adj, reff = [], [], [], []
        Kt_a, Kt_b, Kc_a, Kc_b = [], [], [], []
        ia, ib = half_idx if half_idx is not None else (None, None)
        for X in layers:
            K = X @ X.T
            c = _center(K); Kc.append(c.flatten())
            Kd = X.double() @ X.double().T                                     # fp64: unbiased HSIC cancels large terms
            t = Kd.clone(); t.fill_diagonal_(0); Kt.append(t.flatten())
            reff.append((torch.trace(c) ** 2 / (c * c).sum()).item())
            S = K.clone(); S.fill_diagonal_(-1e8)
            idx = S.topk(k, dim=1).indices
            A = torch.zeros(n, n, device=X.device); A.scatter_(1, idx, 1.0); adj.append(A.flatten())
            if ia is not None:
                for sub, Kt_s, Kc_s in ((ia, Kt_a, Kc_a), (ib, Kt_b, Kc_b)):
                    Ks = K[sub][:, sub]; Kc_s.append(_center(Ks).flatten())
                    ts = Kd[sub][:, sub].clone(); ts.fill_diagonal_(0); Kt_s.append(ts.flatten())
        self.Kc = torch.stack(Kc); self.Kc_norm = self.Kc.norm(dim=1)
        self.Kt = torch.stack(Kt); self.adj = torch.stack(adj)
        self.reff = np.array(reff)
        self.hu = self._hsic_parts(self.Kt, n)
        if ia is not None:
            self.Kc_a = torch.stack(Kc_a); self.Kc_b = torch.stack(Kc_b)
            self.Kc_a_norm = self.Kc_a.norm(dim=1); self.Kc_b_norm = self.Kc_b.norm(dim=1)
            self.hu_a = self._hsic_parts(torch.stack(Kt_a), len(ia))
            self.hu_b = self._hsic_parts(torch.stack(Kt_b), len(ib))

    @staticmethod
    def _hsic_parts(Kt, n):
        M = Kt.view(Kt.shape[0], n, n)
        return dict(flat=Kt, tot=Kt.sum(1), rows=M.sum(2), n=n)

def _hsic_u_pairs(A, B):
    """Unbiased HSIC (Song et al. 2012) for all layer pairs. A, B: dicts from _hsic_parts."""
    n = A["n"]
    t1 = A["flat"] @ B["flat"].T
    t2 = A["tot"][:, None] * B["tot"][None, :] / ((n - 1) * (n - 2))
    t3 = 2.0 * (A["rows"] @ B["rows"].T) / (n - 2)
    return (t1 + t2 - t3) / (n * (n - 3))

def _hsic_u_self(A):
    n = A["n"]
    return ((A["flat"] ** 2).sum(1) + A["tot"] ** 2 / ((n - 1) * (n - 2))
            - 2.0 * (A["rows"] ** 2).sum(1) / (n - 2)) / (n * (n - 3))

def unbiased_cka_pairs(A, B, rel_tol=1e-6):
    """fp64. Layers whose self-HSIC is ~0 or negative (near-constant representations, e.g. a ViT's
    first-block CLS token) are undefined for CKA and returned as NaN instead of dividing by ~0."""
    sa, sb = _hsic_u_self(A), _hsic_u_self(B)
    va = sa > rel_tol * sa.max().clamp_min(1e-300); vb = sb > rel_tol * sb.max().clamp_min(1e-300)
    u = _hsic_u_pairs(A, B) / torch.sqrt(sa.clamp_min(1e-300)[:, None] * sb.clamp_min(1e-300)[None, :])
    u[~va, :] = float("nan"); u[:, ~vb] = float("nan")
    return u.float()

def _nanargmax2(M):
    i = int(torch.nan_to_num(M, nan=-float("inf")).argmax()); return divmod(i, M.shape[1])

def linear_cka_pairs(x, y):
    return (x.Kc @ y.Kc.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])

def mknn_pairs(x, y):
    return (x.adj @ y.adj.T) / (x.n * x.k)

def _perm_flat_index(p, n):
    return (p[:, None] * n + p[None, :]).flatten()

def calibrate(obs, null, alpha=0.05, smax=1.0):
    K = len(null)
    allv = np.sort(np.append(null, obs))
    tau = allv[int(math.ceil((1 - alpha) * (K + 1))) - 1]
    cal = max((obs - tau) / (smax - tau), 0.0) if smax > tau else 0.0
    p = (1 + np.sum(null >= obs)) / (K + 1)
    return cal, tau, p

def align_pair(x, y, perms, alpha=0.05, perms_half=None):
    """x: language SideStats, y: vision SideStats. Returns dict of scalar results + matrices."""
    n = x.n
    cka = linear_cka_pairs(x, y); ucka = unbiased_cka_pairs(x.hu, y.hu); mk = mknn_pairs(x, y)
    i_c, j_c = divmod(int(cka.argmax()), cka.shape[1])
    i_u, j_u = _nanargmax2(ucka)
    i_m, j_m = divmod(int(mk.argmax()), mk.shape[1])
    null_cka_max, null_mk_max, null_cka_fixed, null_arg = [], [], [], []
    for p in perms:
        fi = _perm_flat_index(p, n)
        Kc_p = y.Kc.index_select(1, fi); A_p = y.adj.index_select(1, fi)
        s = (x.Kc @ Kc_p.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])
        a_ = int(s.argmax()); null_arg.append(a_)
        null_cka_max.append(s.view(-1)[a_].item()); null_cka_fixed.append(s[i_c, j_c].item())
        null_mk_max.append(((x.adj @ A_p.T) / (n * x.k)).max().item())
    cka_max = cka.max().item(); mk_max = mk.max().item()
    cal_agg, tau_agg, p_agg = calibrate(cka_max, np.array(null_cka_max), alpha)
    cal_fix, tau_fix, _ = calibrate(cka_max, np.array(null_cka_fixed), alpha)
    mk_cal, mk_tau, mk_p = calibrate(mk_max, np.array(null_mk_max), alpha)
    out = dict(cka_max=cka_max, cka_cal=cal_agg, cka_tau=tau_agg, cka_p=p_agg,
               cka_cal_fixedpair=cal_fix, cka_tau_fixedpair=tau_fix,
               cka_null_mean=float(np.mean(null_cka_max)),
               ucka_max=float(np.nanmax(ucka.cpu().numpy())), mknn_max=mk_max, mknn_cal=mk_cal, mknn_tau=mk_tau, mknn_p=mk_p,
               best_cka=(i_c, j_c), best_ucka=(i_u, j_u), best_mknn=(i_m, j_m),
               reff_x=float(x.reff[i_c]), reff_y=float(y.reff[j_c]))
    out["pred_shrink"] = 1.0 / math.sqrt((1 + out["reff_x"] / n) * (1 + out["reff_y"] / n))
    # which layer pair sets the aggregate null threshold most often, and how low-rank is it?
    mode = max(set(null_arg), key=null_arg.count); im, jm = divmod(mode, cka.shape[1])
    out.update(nullmax_pair=(im, jm), nullmax_pair_freq=null_arg.count(mode) / len(null_arg),
               nullmax_reff_x=float(x.reff[im]), nullmax_reff_y=float(y.reff[jm]))
    if hasattr(x, "hu_a") and hasattr(y, "hu_a"):
        ua = unbiased_cka_pairs(x.hu_a, y.hu_a)
        ia, ja = _nanargmax2(ua)
        ub = unbiased_cka_pairs(x.hu_b, y.hu_b)
        out["ucka_heldout"] = ub[ia, ja].item(); out["best_heldout"] = (ia, ja)
        # held-out CALIBRATED CKA: pick the layer pair on half A by UNBIASED CKA (raw CKA is inflated by the
        # chance offset and would favour high-rank layers), then calibrate that single pair's raw CKA on half B
        ic, jc = ia, ja
        xb, yb = x.Kc_b[ic], y.Kc_b[jc]; nb = int(round(math.sqrt(xb.numel())))
        obs = (xb @ yb / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
        nulls = [(xb @ yb.index_select(0, _perm_flat_index(p, nb)) / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
                 for p in (perms_half or [])]
        if nulls:
            c_, t_, p_ = calibrate(obs, np.array(nulls), alpha)
            out.update(cka_heldout_raw=obs, cka_heldout_cal=c_, cka_heldout_tau=t_, best_heldout_cka=(ic, jc))
    mats = dict(cka=cka.cpu().numpy(), ucka=ucka.cpu().numpy(), mknn=mk.cpu().numpy())
    return out, mats

def layer_diagnostics(feats):
    """Outlier diagnostics on raw (un-normalised) features. feats: N x L x D."""
    rows = []
    for l in range(feats.shape[1]):
        f = feats[:, l, :]
        v = f.var(0); vs = torch.sort(v, descending=True).values; tot = v.sum()
        absf = f.abs()
        fc = f - f.mean(0, keepdim=True)
        G = fc @ fc.T
        rows.append(dict(layer=l, top1_var_share=(vs[0] / tot).item(), top10_var_share=(vs[:10].sum() / tot).item(),
                         max_over_median_abs=(absf.max() / absf.median().clamp_min(1e-12)).item(),
                         reff_raw=(torch.trace(G) ** 2 / (G * G).sum()).item()))
    return rows

In [ ]:
# ============================== EXTRACTION CODE ==============================
# ===================== Feature extraction (Huh et al. protocol) =====================
# Language: every hidden state (embeddings + each block), mean-pooled over non-pad tokens,
#           left padding to the longest caption in the dataset, bf16 weights.
# Vision:   CLS token after every transformer block (= Huh's "blocks.{i}.add_1" node), timm transforms.
import gc, math, os, shutil, time
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def _dtype_kw(dtype):
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}

def load_tokenizer(name):
    try:
        tok = AutoTokenizer.from_pretrained(name)
    except Exception as e:                      # e.g. OpenLLaMA fast-tokenizer conversion issues
        print("  fast tokenizer failed, using slow:", repr(e)[:120])
        tok = AutoTokenizer.from_pretrained(name, use_fast=False)
    if "huggyllama" in name:
        tok.pad_token = tok.unk_token           # Huh et al. set "[PAD]" (not in vocab); padded positions are masked either way
    elif tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return tok

def load_llm(name, four_bit=False):
    q = None
    if four_bit:                                # Huh et al.'s --qlora setting
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                               bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    return AutoModelForCausalLM.from_pretrained(name, device_map="auto", quantization_config=q,
                                                **_dtype_kw(torch.bfloat16)).eval()

@torch.no_grad()
def extract_llm(name, texts, batch_size=8, four_bit=False):
    tok = load_tokenizer(name)
    model = load_llm(name, four_bit)
    n_params = sum(p.numel() for p in model.parameters())
    if four_bit:
        n_params = {"huggyllama/llama-65b": 65.2e9}.get(name, n_params)
    tokens = tok(texts, padding="longest", return_tensors="pt")
    dev = next(model.parameters()).device
    feats, losses, bpbs = [], [], []
    crit = torch.nn.CrossEntropyLoss(reduction="none")
    for i in range(0, len(texts), batch_size):
        ids = tokens["input_ids"][i:i + batch_size].to(dev).long()
        am = tokens["attention_mask"][i:i + batch_size].to(dev).long()
        out = model(input_ids=ids, attention_mask=am, output_hidden_states=True)
        m = am[:, :-1].float()
        tok_loss = m * crit(out.logits[:, :-1].float().permute(0, 2, 1), ids[:, 1:])
        losses.append((tok_loss.sum(-1) / m.sum(-1)).cpu())
        nbytes = torch.tensor([len(t.encode("utf-8")) for t in texts[i:i + batch_size]], dtype=torch.float32)
        bpbs.append((tok_loss.sum(-1).cpu() * math.log2(math.e)) / nbytes)
        w = am.unsqueeze(-1).to(out.hidden_states[0].dtype)
        pooled = [((h * w).sum(1) / w.sum(1)).float().cpu() for h in out.hidden_states]  # per layer: B x D
        feats.append(torch.stack(pooled, 1).to(torch.bfloat16))                         # B x L x D
        del out
    res = dict(feats=torch.cat(feats), num_params=int(n_params),
               loss=torch.cat(losses).mean().item(), bpb=torch.cat(bpbs).mean().item())
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

@torch.no_grad()
def extract_vision(name, images, batch_size=64, pretrained=True):
    import timm
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform
    model = timm.create_model(name, pretrained=pretrained).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; model = model.to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    tf = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
    store = {}
    hooks = [blk.register_forward_hook(lambda m, i, o, k=k: store.__setitem__(k, o[:, 0, :].float().cpu()))
             for k, blk in enumerate(model.blocks)]
    feats = []
    for i in range(0, len(images), batch_size):
        x = torch.stack([tf(im.convert("RGB")) for im in images[i:i + batch_size]]).to(dev)
        store.clear(); model(x)
        feats.append(torch.stack([store[k] for k in range(len(model.blocks))], 1))
    for h in hooks: h.remove()
    res = dict(feats=torch.cat(feats).to(torch.bfloat16), num_params=int(n_params))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

def clear_hf_cache(name):
    hub = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
    shutil.rmtree(os.path.join(hub, "models--" + name.replace("/", "--")), ignore_errors=True)

In [ ]:
# ============================== DATA: WIT-1024 (Huh et al.) ==============================
if DRYRUN:
    from dryrun_support import make_dataset
    images, texts = make_dataset()
else:
    from datasets import load_dataset
    ds = load_dataset("minhuh/prh", revision="wit_1024", split="train")
    texts = [str(x["text"][0]) for x in ds]
    images = [x["image"] for x in ds]
N = len(texts)
log(f"dataset: {N} image-caption pairs; example caption: {texts[0][:100]!r}")

def feat_path(name):
    return os.path.join(FEAT_DIR, name.replace("/", "_") + ".pt")

In [ ]:
# ============================== EXTRACT VISION FEATURES ==============================
for name in VISION:
    fp = feat_path(name)
    if os.path.exists(fp):
        log("vision exists, skipping:", name); continue
    t0 = time.time()
    try:
        r = extract_vision(name, images, batch_size=64, pretrained=not DRYRUN)
        torch.save(r, fp)
        log(f"vision done: {name} feats={tuple(r['feats'].shape)} ({time.time()-t0:.0f}s)")
    except Exception as e:
        log("VISION FAILED:", name, repr(e)[:300])

In [ ]:
# ============================== EXTRACT LLM FEATURES ==============================
# Small models first, so a failure on the largest ones never costs the rest.
for name in LLMS:
    fp = feat_path(name)
    if os.path.exists(fp):
        log("llm exists, skipping:", name); continue
    need = APPROX_GB.get(name, 15) + 10
    free = shutil.disk_usage("/").free / 1e9
    if not DRYRUN and free < need:
        log(f"SKIPPING {name}: needs ~{need} GB free disk, have {free:.0f} GB"); continue
    t0 = time.time()
    try:
        r = extract_llm(name, texts, batch_size=llm_batch_size(name), four_bit=name in FOUR_BIT)
        torch.save(r, fp)
        log(f"llm done: {name} feats={tuple(r['feats'].shape)} loss={r['loss']:.3f} bpb={r['bpb']:.3f} ({time.time()-t0:.0f}s)")
    except Exception as e:
        log("LLM FAILED:", name, repr(e)[:300])
    finally:
        if not DRYRUN:
            clear_hf_cache(name)
        torch.cuda.empty_cache()

In [ ]:
# ============================== OUTLIER / SPECTRUM DIAGNOSTICS ==============================
diag_fp = os.path.join(RES_DIR, "diagnostics.csv")
rows = []
for name, modality in [(m, "language") for m in LLMS] + [(m, "vision") for m in VISION]:
    if not os.path.exists(feat_path(name)): continue
    f = torch.load(feat_path(name))["feats"].to(DEV).float()
    for r in layer_diagnostics(f):
        rows.append(dict(model=name, modality=modality, n_layers=f.shape[1], **r))
    for v in VARIANTS:                                   # effective rank of what the metrics actually see
        for l, X in enumerate(preprocess(f, v)):
            Xc = X - X.mean(0, keepdim=True); G = Xc @ Xc.T
            rows.append(dict(model=name, modality=modality, layer=l, variant=v,
                             reff=(torch.trace(G) ** 2 / (G * G).sum()).item()))
    del f; torch.cuda.empty_cache()
diag = pd.DataFrame(rows); diag.to_csv(diag_fp, index=False)
log("diagnostics saved:", diag_fp, len(diag), "rows")

In [ ]:
# ============================== ALIGNMENT: every pair, every layer pair, 3 preprocessings ==============================
g = torch.Generator().manual_seed(SEED)
PERMS = [torch.randperm(N, generator=g).to(DEV) for _ in range(N_PERMS)]
_split = torch.randperm(N, generator=g)
HALF = (_split[: N // 2].to(DEV), _split[N // 2:].to(DEV))
PERMS_HALF = [torch.randperm(N - N // 2, generator=g).to(DEV) for _ in range(N_PERMS)]
ENGINE_VERSION = "v3"   # v3: fp64 unbiased CKA, NaN for degenerate layers, held-out calibrated CKA (selected by unbiased CKA)
avail_llm = [m for m in LLMS if os.path.exists(feat_path(m))]
avail_vis = [m for m in VISION if os.path.exists(feat_path(m))]
log(f"aligning {len(avail_llm)} LLMs x {len(avail_vis)} vision models x {len(VARIANTS)} variants")

def side_stats(name, variant):
    f = torch.load(feat_path(name))["feats"].to(DEV).float()
    s = SideStats(preprocess(f, variant), k=TOPK, half_idx=HALF)
    del f; return s

for variant in VARIANTS:
    todo = [m for m in avail_llm if not os.path.exists(os.path.join(RES_DIR, f"align_{ENGINE_VERSION}_{variant}_{m.replace('/', '_')}.csv"))]
    if not todo:
        log("variant done:", variant); continue
    t0 = time.time()
    vis_stats = {v: side_stats(v, variant) for v in avail_vis}
    log(f"[{variant}] vision stats built ({time.time()-t0:.0f}s)")
    for lm in todo:
        t1 = time.time(); xs = side_stats(lm, variant); rows, mats = [], {}
        for vm, ys in vis_stats.items():
            out, m = align_pair(xs, ys, PERMS, alpha=ALPHA, perms_half=PERMS_HALF)
            rows.append(dict(variant=variant, llm=lm, vision=vm, **out))
            for k_, a in m.items():
                mats[f"{vm}__{k_}"] = a
        pd.DataFrame(rows).to_csv(os.path.join(RES_DIR, f"align_{ENGINE_VERSION}_{variant}_{lm.replace('/', '_')}.csv"), index=False)
        np.savez_compressed(os.path.join(MAT_DIR, f"{ENGINE_VERSION}_{variant}_{lm.replace('/', '_')}.npz"), **mats)
        log(f"[{variant}] {lm}: {len(rows)} pairs ({time.time()-t1:.0f}s)")
        del xs; torch.cuda.empty_cache()
    del vis_stats; torch.cuda.empty_cache()

In [ ]:
# ============================== ANALYSIS ==============================
import matplotlib; import matplotlib.pyplot as plt
res = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(RES_DIR, f"align_{ENGINE_VERSION}_*.csv"))], ignore_index=True)
meta = []
for m in avail_llm:
    d = torch.load(feat_path(m))
    meta.append(dict(llm=m, bpb=d["bpb"], loss=d["loss"], llm_params=d["num_params"],
                     llm_family=m.split("/")[1].split("-")[0].split("_")[0]))
meta = pd.DataFrame(meta); meta.to_csv(os.path.join(RES_DIR, "llm_meta.csv"), index=False)
def vfam(v):
    return ("clip_ft" if "ft_in12k" in v else "clip" if "clip" in v else "dinov2" if "dinov2" in v
            else "mae" if ".mae" in v else "in21k")
res = res.merge(meta, on="llm"); res["vision_family"] = res.vision.map(vfam)
# predicted aggregate threshold: chance offset of the highest-effective-rank layer on each side
_rmax = diag[diag.variant.notna()].groupby(["model", "variant"]).reff.max().reset_index()
res = (res.merge(_rmax.rename(columns={"model": "llm", "reff": "rmax_x"}), on=["llm", "variant"])
          .merge(_rmax.rename(columns={"model": "vision", "reff": "rmax_y"}), on=["vision", "variant"]))
res["tau_pred"] = np.sqrt(res.rmax_x * res.rmax_y) / N
res.to_csv(os.path.join(RES_DIR, "alignment_all.csv"), index=False)

METRICS = ["cka_max", "cka_cal", "cka_heldout_cal", "ucka_max", "ucka_heldout", "mknn_max", "mknn_cal"]
LABEL = dict(cka_max="CKA (raw, Huh)", cka_cal="CKA calibrated (Aristotelian)", cka_heldout_cal="CKA calibrated (held-out layer)", ucka_max="Unbiased CKA (max)",
             ucka_heldout="Unbiased CKA (held-out layer)", mknn_max="Mutual k-NN (raw)", mknn_cal="Mutual k-NN calibrated")
from scipy.stats import pearsonr, spearmanr
trend = []
for v in VARIANTS:
    for met in METRICS:
        d = res[res.variant == v]
        if met not in d or len(d.llm.unique()) < 3: continue
        rs = [pearsonr(-g_.bpb, g_[met])[0] for _, g_ in d.groupby("vision") if g_[met].std() > 0]
        ss = [spearmanr(-g_.bpb, g_[met])[0] for _, g_ in d.groupby("vision") if g_[met].std() > 0]
        mean_by_llm = d.groupby("llm")[met].mean().reindex(meta.sort_values("bpb", ascending=False).llm)
        trend.append(dict(variant=v, metric=met, pearson_vs_capability=np.nanmean(rs) if rs else np.nan,
                          spearman_vs_capability=np.nanmean(ss) if ss else np.nan,
                          mean_score=d[met].mean(), weakest_llm_score=mean_by_llm.iloc[0], strongest_llm_score=mean_by_llm.iloc[-1]))
trend = pd.DataFrame(trend); trend.to_csv(os.path.join(RES_DIR, "trend_table.csv"), index=False)
print(trend.round(3).to_string())

# Figure 1: mean score (over vision models) vs LLM capability, every metric x variant
fig, axs = plt.subplots(len(VARIANTS), len(METRICS), figsize=(3.1 * len(METRICS), 2.6 * len(VARIANTS)), squeeze=False)
for i, v in enumerate(VARIANTS):
    d = res[res.variant == v].groupby(["llm", "bpb", "llm_family"])[METRICS].mean().reset_index()
    for j, met in enumerate(METRICS):
        ax = axs[i, j]
        for fam, gf in d.groupby("llm_family"):
            gf = gf.sort_values("bpb"); ax.plot(gf.bpb, gf[met], "o-", ms=3, label=fam)
        ax.invert_xaxis(); ax.grid(alpha=.3)
        if i == 0: ax.set_title(LABEL[met], fontsize=8)
        if j == 0: ax.set_ylabel(f"{v}\nscore", fontsize=8)
        if i == len(VARIANTS) - 1: ax.set_xlabel("LLM bits/byte (better →)", fontsize=8)
axs[0, 0].legend(fontsize=6)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_trends.png"), dpi=130); plt.show()

# Figure 2: effective rank / n and outlier share per LLM (what decides the E1 calibration penalty)
dl = diag[(diag.modality == "language") & diag.variant.isna()].merge(meta, left_on="model", right_on="llm")
dr = diag[diag.variant.notna()].copy()
fig, axs = plt.subplots(1, 4, figsize=(17, 3.3))
for v in VARIANTS:
    s = dr[(dr.variant == v) & (dr.modality == "language")].groupby("model").reff.median().reindex(meta.llm)
    axs[0].plot(meta.bpb, s / N, "o", label=v)
axs[0].set_title("LLM effective rank / n (median over layers)", fontsize=9); axs[0].invert_xaxis(); axs[0].legend(fontsize=7)
s = dl.groupby("model").top1_var_share.max().reindex(meta.llm)
axs[1].plot(meta.bpb, s, "o"); axs[1].set_title("LLM: max over layers of top-1 dim variance share", fontsize=9); axs[1].invert_xaxis()
d = res[res.variant == "huh"]
dm = d.groupby(["llm", "bpb"])[["cka_tau", "cka_tau_fixedpair", "cka_heldout_tau"]].mean().reset_index()
for c_, lab in [("cka_tau", "aggregate (max over all layer pairs)"), ("cka_tau_fixedpair", "selected pair only"), ("cka_heldout_tau", "held-out pair (n/2)")]:
    axs[2].plot(dm.bpb, dm[c_], "o", label=lab)
axs[2].set_xlabel("LLM bits/byte (better →)"); axs[2].invert_xaxis(); axs[2].set_ylabel("95% chance threshold τ")
axs[2].set_title("Calibration thresholds (huh variant)", fontsize=9); axs[2].legend(fontsize=7)
axs[3].scatter(res.tau_pred, res.cka_tau, s=8); lim = [0, max(res.tau_pred.max(), res.cka_tau.max()) * 1.05]
axs[3].plot(lim, lim, "k:"); axs[3].set_xlabel("√(max-rank LLM layer × max-rank vision layer) / n"); axs[3].set_ylabel("observed aggregate τ")
axs[3].set_title("Threshold set by highest-rank layers?", fontsize=9)
for a in axs[:2]: a.set_xlabel("LLM bits/byte (better →)")
for a in axs: a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_effrank_outliers.png"), dpi=130); plt.show()

In [ ]:
# ============================== SUMMARY + PACKAGE FOR CLAUDE ==============================
summary = dict(
    run="smoke" if SMOKE_TEST else "full", n=N, n_perms=N_PERMS, llms=avail_llm, vision=avail_vis,
    engine=ENGINE_VERSION, trend=trend.round(4).to_dict("records"),
    tau_pred_fit=dict(corr=float(np.corrcoef(res.tau_pred, res.cka_tau)[0, 1]), mae=float((res.tau_pred - res.cka_tau).abs().mean())),
    tau_vs_capability=res[res.variant == "huh"].groupby("llm")[["bpb", "cka_tau", "tau_pred", "rmax_x"]].mean().round(4).reset_index().to_dict("records"),
    thresholds=res.groupby("variant")[["cka_tau", "cka_tau_fixedpair", "cka_heldout_tau", "nullmax_pair_freq", "nullmax_reff_x", "nullmax_reff_y", "reff_x", "reff_y"]].median().round(4).reset_index().to_dict("records"),
    reff_over_n=dr.groupby(["modality", "variant"]).reff.describe()[["min", "50%", "max"]].div(N).round(4).reset_index().to_dict("records"),
    llm_top1_var_share=dl.groupby("model").top1_var_share.max().round(4).to_dict(),
    pred_shrink=res.groupby("variant").pred_shrink.describe()[["min", "50%", "max"]].round(4).reset_index().to_dict("records"),
    failures=[l.strip() for l in open(LOG) if "FAILED" in l or "SKIPPING" in l],
)
json.dump(summary, open(os.path.join(RES_DIR, "summary.json"), "w"), indent=1, default=str)
zip_base = os.path.join(RUN, "prh_audit_results")
import zipfile
with zipfile.ZipFile(zip_base + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("results", "matrices", "figs"):
        for f in glob.glob(os.path.join(RUN, sub, "*")):
            if sub != "figs" and not os.path.basename(f).startswith(("diagnostics", "llm_meta", "alignment_all", "trend", "summary", ENGINE_VERSION, f"align_{ENGINE_VERSION}")): continue   # current engine only
            z.write(f, os.path.join(sub, os.path.basename(f)))
    z.write(LOG, "log.txt")
log("packaged:", zip_base + ".zip", f"{os.path.getsize(zip_base + '.zip')/1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(zip_base + ".zip")
print(json.dumps({k: summary[k] for k in ("failures",)}, indent=1))